<a href="https://colab.research.google.com/github/mahbubpatwari/DemoApp/blob/main/lastPart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/results/langratio_robustness', exist_ok=True)

!pip -q install nltk
import nltk
nltk.download('words')

[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


True

In [ ]:
from google.colab import files
files.upload()   # pick test_with_preds.csv (the file with Sentence, Label, clean_text, pred)
!mv test_with_preds.csv /content/data/test_with_preds.csv

import pandas as pd
df = pd.read_csv('/content/data/test_with_preds.csv')
print(df.shape, df.columns.tolist())
print(df['Label'].value_counts())

Saving test_with_preds.csv to test_with_preds.csv
(2997, 4) ['Sentence', 'Label', 'clean_text', 'pred']
Label
2    991
1    926
0    800
3    280
Name: count, dtype: int64


In [ ]:
from nltk.corpus import words as nltk_words
import string

V_EN = set(w.lower() for w in nltk_words.words())

def rho_v1(text):
    tokens = str(text).split()
    if not tokens:
        return 0.0
    eng = sum(1 for t in tokens if t.lower() in V_EN)
    return eng / len(tokens)

df['rho_v1'] = df['clean_text'].apply(rho_v1)

def stratum(r, hi=0.60, lo=0.25):
    if r >= hi: return 'English-heavy'
    elif r <= lo: return 'Bangla-heavy'
    else: return 'Balanced'

df['stratum_v1'] = df['rho_v1'].apply(lambda r: stratum(r))

print(df['stratum_v1'].value_counts())


stratum_v1
Balanced         1785
Bangla-heavy      786
English-heavy     426
Name: count, dtype: int64


In [ ]:
import re

# Small, deliberately conservative list of romanised Bengali function words that
# collide with valid English dictionary entries (the "lexical collision" bias).
# Extend this list based on manual inspection of your own false-positive tokens
# if you have time — even this short version removes the worst offenders.
BANGLA_STOPWORDS = {
    'ta', 'ti', 'na', 'ni', 'ei', 'oi', 'oi', 'ar', 'or', 'e', 'te',
    'ache', 'ase', 'hoy', 'hobe', 'hoise', 'lage', 'dey', 'de', 'jai',
    'ke', 'k', 'ba', 'ei', 'oy', 'ho', 'nai', 'na', 'ar', 'r'
}

def rho_v2(text):
    tokens = str(text).split()
    # fix 1: strip leading/trailing punctuation before lookup
    cleaned = [t.strip(string.punctuation) for t in tokens]
    # fix 2: exclude pure numerals
    cleaned = [t for t in cleaned if t and not t.isdigit()]
    if not cleaned:
        return 0.0
    # fix 3: subtract known Bangla-English collision words from the English test
    eng = sum(1 for t in cleaned
               if t.lower() in V_EN and t.lower() not in BANGLA_STOPWORDS)
    return eng / len(cleaned)

df['rho_v2'] = df['clean_text'].apply(rho_v2)
df['stratum_v2'] = df['rho_v2'].apply(lambda r: stratum(r))

print(df['stratum_v2'].value_counts())
print("\nHow many samples changed stratum:",
      (df['stratum_v1'] != df['stratum_v2']).sum(), "/", len(df))

stratum_v2
Balanced         1639
Bangla-heavy     1035
English-heavy     323
Name: count, dtype: int64

How many samples changed stratum: 549 / 2997


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

def stratum_metrics(data, stratum_col):
    rows = []
    for s in ['English-heavy', 'Balanced', 'Bangla-heavy']:
        sub = data[data[stratum_col] == s]
        n = len(sub)
        mixed_n = (sub['Label'] == 3).sum()
        if n == 0:
            rows.append([s, n, mixed_n, None, None, None])
            continue
        acc = accuracy_score(sub['Label'], sub['pred'])
        macro_f1 = f1_score(sub['Label'], sub['pred'], average='macro', labels=[0,1,2,3])
        _, _, f1_per, _ = precision_recall_fscore_support(
            sub['Label'], sub['pred'], labels=[0,1,2,3], zero_division=0)
        rows.append([s, n, mixed_n, round(acc,3), round(macro_f1,3), round(f1_per[3],3)])
    return pd.DataFrame(rows, columns=['Stratum','n','Mixed_n','Accuracy','MacroF1','MixedF1'])

results_v1 = stratum_metrics(df, 'stratum_v1')
results_v2 = stratum_metrics(df, 'stratum_v2')

print("=== Original estimator (v1) ===")
print(results_v1)
print("\n=== Corrected estimator (v2) ===")
print(results_v2)

=== Original estimator (v1) ===
         Stratum     n  Mixed_n  Accuracy  MacroF1  MixedF1
0  English-heavy   426       69     0.716    0.722    0.778
1       Balanced  1785      161     0.708    0.670    0.495
2   Bangla-heavy   786       50     0.740    0.685    0.474

=== Corrected estimator (v2) ===
         Stratum     n  Mixed_n  Accuracy  MacroF1  MixedF1
0  English-heavy   323       51     0.768    0.764    0.812
1       Balanced  1639      160     0.697    0.669    0.532
2   Bangla-heavy  1035       69     0.735    0.673    0.429


In [ ]:
def stratum_alt(r, hi=0.50, lo=0.30):
    if r >= hi: return 'English-heavy'
    elif r <= lo: return 'Bangla-heavy'
    else: return 'Balanced'

df['stratum_altthresh'] = df['rho_v1'].apply(lambda r: stratum_alt(r))
results_altthresh = stratum_metrics(df, 'stratum_altthresh')

print("=== Original estimator, alternative thresholds (0.50 / 0.30) ===")
print(results_altthresh)

=== Original estimator, alternative thresholds (0.50 / 0.30) ===
         Stratum     n  Mixed_n  Accuracy  MacroF1  MixedF1
0  English-heavy   938      121     0.715    0.710    0.681
1       Balanced  1046       92     0.704    0.667    0.500
2   Bangla-heavy  1013       67     0.734    0.669    0.413


In [ ]:
def stratum_alt(r, hi=0.50, lo=0.30):
    if r >= hi: return 'English-heavy'
    elif r <= lo: return 'Bangla-heavy'
    else: return 'Balanced'

df['stratum_altthresh'] = df['rho_v1'].apply(lambda r: stratum_alt(r))
results_altthresh = stratum_metrics(df, 'stratum_altthresh')

print("=== Original estimator, alternative thresholds (0.50 / 0.30) ===")
print(results_altthresh)

=== Original estimator, alternative thresholds (0.50 / 0.30) ===
         Stratum     n  Mixed_n  Accuracy  MacroF1  MixedF1
0  English-heavy   938      121     0.715    0.710    0.681
1       Balanced  1046       92     0.704    0.667    0.500
2   Bangla-heavy  1013       67     0.734    0.669    0.413


In [ ]:
!zip -r /content/langratio_robustness.zip /content/results/langratio_robustness
files.download('/content/langratio_robustness.zip')

  adding: content/results/langratio_robustness/ (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>